In [1]:
import numpy as np
import pandas as pd

In [2]:
from cvxopt import solvers, matrix

# 2025/7/8 (schedual) (total invest = 320000) 
## stock: 2330, 2308, 2357, 2897

In [3]:
##2019/10 to 2025/6 (monthly)
data_1 = pd.read_csv("portfolio/pratical use/2308.TW.csv")
data_2 = pd.read_csv("portfolio/pratical use/2330.TW.csv")
data_3 = pd.read_csv("portfolio/pratical use/2357.TW.csv")
data_4 = pd.read_csv("portfolio/pratical use/2897.TW.csv")

stock_index = ['2308', '2330', '2357', '2897']
portfolio = [data_1, data_2, data_3, data_4]

In [4]:
##delete NA and cauculate monthly return
for data in portfolio:
    data.dropna(inplace=True)
    monthly_return = [0]
    for i in range(0, len(data)-1):
        monthly_return.append((data.iat[i+1, 5]/data.iat[i, 5] - 1) * 100)
    data['Monthly Return (%)'] = monthly_return
    data.drop([0], inplace = True) ##drop first data which its return is 0

In [5]:
##statistics of each stock in portfolio
statistics = []
for data in portfolio:
    r = data['Monthly Return (%)']
    statistics.append([r.mean(), r.var(), r.std(), r.skew(), r.kurt()])
    
statistics_df = pd.DataFrame(statistics)
statistics_df.index = stock_index
statistics_df.columns = ["Mean", "Variance", "Std", "Skewness", "Kurtosis"]
statistics_df

,Mean,Variance,Std,Skewness,Kurtosis
2308,2.206547,76.351566,8.737938,0.505657,-0.575852
2330,2.417162,77.654058,8.812154,0.900316,3.056718
2357,2.555348,70.040945,8.369047,0.662800,0.081937
2897,0.700329,26.200085,5.118602,0.374156,1.022980


In [6]:
##collect the reutrn data of each stock in portfolio
return_data = pd.concat([data_1['Monthly Return (%)'], data_2['Monthly Return (%)'], 
                         data_3['Monthly Return (%)'], data_4['Monthly Return (%)']], axis = 1)
return_data.index = data_1['Date'].tolist()
return_data.columns = stock_index
return_data

,2308,2330,2357,2897
2019/11/1,4.480930,2.176205,11.619924,1.041656
2019/12/1,8.209200,8.526789,0.438396,0.644333
2020/1/1,-5.604720,-2.617819,-3.244300,-2.304749
2020/2/1,-2.099359,-1.248213,-8.032588,-1.310606
2020/3/1,-13.930267,-13.293083,-0.732118,-13.811423
...,...,...,...,...
2025/2/1,-8.114459,-8.369256,12.354550,0.100503
2025/3/1,-10.446363,-12.500242,-10.703223,-1.104418
2025/4/1,-7.361963,0.245093,-4.761658,-3.654822
2025/5/1,12.143315,6.047357,7.757824,-0.316122


In [7]:
##cauculate covariance
cov = return_data.cov()
cov

,2308,2330,2357,2897
2308,76.351566,54.429797,10.055982,11.660307
2330,54.429797,77.654058,12.979937,11.364501
2357,10.055982,12.979937,70.040945,9.882719
2897,11.660307,11.364501,9.882719,26.200085


In [8]:
##cauculate correlation
corr = return_data.corr()
corr

,2308,2330,2357,2897
2308,1.000000,0.706880,0.137512,0.260705
2330,0.706880,1.000000,0.176001,0.251951
2357,0.137512,0.176001,1.000000,0.230701
2897,0.260705,0.251951,0.230701,1.000000


### 用MVP找當期投資比例
#### (改日期、上期數量、目標報酬)

In [9]:
##find the optimal solution of portfolio for min risk(variance) with a target return(monthly)
def ns_mvp(r, mu_targ):
    n = r.shape[1]
    mux = np.array(r.mean()).reshape(n, 1)
    covx = matrix(r.cov().values.tolist())
    onex = np.ones((n, 1))
   
    G = matrix(np.diag(np.ones(n) * -1))
    h0 = matrix(np.repeat(0, n).reshape(n, 1).astype('float'))
    A = matrix(np.hstack((mux, onex)).T)
    b0 = matrix(np.array([mu_targ, 1]).reshape(2, 1))
    q0 = matrix(np.repeat(0, n).reshape(n, 1).astype('float'))
    
    ##quadratic programming
    ##minimize ((1/2 x^T P x) + (q^T x)), subject to ((G x) <= h), ((A x) = b)
    return solvers.qp(P = covx, q = q0, G = G, h = h0, A = A, b = b0)

In [22]:
##cauculate last portfolio value to last month end
last_price = [data_1.iat[-1, 5], data_2.iat[-1, 5], data_3.iat[-1, 5], data_4.iat[-1, 5]]
last_amount = [141, 65, 206, 15126]
last_port = np.multiply(last_price, last_amount)
##add new invest $10000
capital = sum(last_port)
mu_targ = 1.8 ##monthly return(%)
print("Date: 2025/07/08")

##cauculate weight
weight = ns_mvp(return_data, mu_targ)['x']
weight = sum(np.array(weight).reshape(1, 4))
print("Weight: ")
print(weight)
print("")

##cauculate the estimate invest
port = np.array(weight * capital)
port = np.around(port)
print("Capital: " + str(capital))
print("Last Invest: " + str(last_port))
print("New Invest:  " + str(port))
add = [a - b for a, b in zip(port, last_port)]
add = np.around(add)
print("Add:         " + str(add))
print("")

##cauculate the risk
risk = np.sqrt(np.dot(weight.T, np.dot(cov, weight)))
print("Risk (Std): ")
print(risk)
print("")
var = ((mu_targ-1.96*risk)/100 + 1)*capital
loss = capital-var
print("VaR(95%): ")
print(var)
print("Loss in 95% Confidience: ")
print(loss)

Date: 2025/07/08
     pcost       dcost       gap    pres   dres
 0:  1.2214e+01  1.1134e+01  1e+00  1e-16  3e+00
 1:  1.2214e+01  1.2203e+01  1e-02  2e-16  3e-02
 2:  1.2214e+01  1.2214e+01  1e-04  2e-16  3e-04
 3:  1.2214e+01  1.2214e+01  1e-06  6e-17  3e-06
 4:  1.2214e+01  1.2214e+01  1e-08  7e-17  3e-08
Optimal solution found.
Weight: 
[0.14251708 0.16464492 0.32470915 0.36812885]

Capital: 391948.28
Last Invest: [ 57214.98  68900.   126220.32 139612.98]
New Invest:  [ 55859.  64532. 127269. 144287.]
Add:         [-1356. -4368.  1049.  4674.]

Risk (Std): 
4.942467719291601

VaR(95%): 
361034.39129797544
Loss in 95% Confidience: 
30913.88870202459


In [23]:
print("last close px: ")
print(last_price)
print("")

total_amount = [a / b for a, b in zip(port, last_price)]
print("estimate total amount: ")
print(total_amount)
print("")

print("last amount: ")
print(last_amount)
print("")

change = [a - b for a, b in zip(total_amount, last_amount)]
print("estimate buy/sell amount: ")
print(change)

last close px: 
[405.78, 1060.0, 612.72, 9.23]

estimate total amount: 
[137.6583370299177, 60.87924528301887, 207.71151586368975, 15632.394366197183]

last amount: 
[141, 65, 206, 15126]

estimate buy/sell amount: 
[-3.3416629700822966, -4.120754716981132, 1.7115158636897547, 506.39436619718253]


In [24]:
##final portfolio data for this month
final_invest_data = [weight, port, total_amount, last_amount, change]
final_invest_data = pd.DataFrame(final_invest_data)
final_invest_data.index = ['weight', 'value', 'total amount', 'last amount', 'change']
final_invest_data.columns = stock_index
final_invest_data

,2308,2330,2357,2897
weight,0.142517,0.164645,0.324709,0.368129
value,55859.000000,64532.000000,127269.000000,144287.000000
total amount,137.658337,60.879245,207.711516,15632.394366
last amount,141.000000,65.000000,206.000000,15126.000000
change,-3.341663,-4.120755,1.711516,506.394366


In [25]:
##save as a new worksheet in practical_invest.xlsx
##mode = "a", append (default "w", write)
with pd.ExcelWriter("portfolio/practical_invest.xlsx", mode = "a", engine = "openpyxl") as writer:
    final_invest_data.to_excel(writer, sheet_name = "20250708")

### 附錄: 用GMVP對照風險與期望報酬

In [19]:
def ns_gmvp(r):
    n = r.shape[1]
    covx = matrix(r.cov().values.tolist())
    onex = np.ones((n, 1))
    
    G = matrix(np.diag(np.ones(n) * -1))
    h0 = matrix(np.repeat(0, n).reshape(n, 1).astype('float'))
    A = matrix(onex.T)
    b0 = matrix(np.array([1]).astype('float'))
    q0 = matrix(np.repeat(0, n).reshape(n, 1).astype('float'))
    
    return solvers.qp(P = covx, q = q0, G = G, h = h0, A = A, b = b0)

In [20]:
print("Date: 2025/07/08")

weight = ns_gmvp(return_data)['x']
weight = sum(np.array(weight).reshape(1, 4))
print("Weight: ")
print(weight)

port = np.array(weight * capital)
print("Invest: ")
print(port)

risk = np.sqrt(np.dot(weight.T, np.dot(cov, weight)))
print("Risk (Std): ")
print(risk)

mux = np.array(return_data.mean()).reshape(4, 1)
r = np.dot(weight, mux)[0]
print("expected return (%): ")
print(r)

Date: 2025/07/08
     pcost       dcost       gap    pres   dres
 0:  1.0400e+01  9.2190e+00  1e+00  3e-16  3e+00
 1:  1.0397e+01  1.0383e+01  1e-02  1e-16  4e-02
 2:  1.0397e+01  1.0397e+01  1e-04  2e-16  4e-04
 3:  1.0397e+01  1.0397e+01  1e-06  1e-16  4e-06
 4:  1.0397e+01  1.0397e+01  1e-08  3e-17  4e-08
Optimal solution found.
Weight: 
[0.09535377 0.07599359 0.17720221 0.65145043]
Invest: 
[ 37373.74555769  29785.55815021  69454.10208286 255334.87420924]
Risk (Std): 
4.560129332823591
expected return (%): 
1.3031340632160693
